In [2]:
import numpy as np
import pandas as pd
from obspy.core import read, UTCDateTime
from obspy.core.inventory import read_inventory
from obspy.core.util.attribdict import AttribDict
from obspy.clients.fdsn import Client
from collections import defaultdict
from obspy.geodetics import gps2dist_azimuth
from obspy.geodetics import kilometers2degrees
from obspy.taup import TauPyModel
import matplotlib.pyplot as plt

In [3]:
# AUSPASS     https://auspass.edu.au
# BGR         https://eida.bgr.de
# BGS         https://eida.bgs.ac.uk
# EARTHSCOPE  https://service.earthscope.org
# EIDA        http://eida-federator.ethz.ch
# EMSC        https://www.seismicportal.eu
# EPOSFR      https://seisdata.epos-france.fr
# ETH         https://eida.ethz.ch
# GEOFON      https://geofon.gfz.de
# GEONET      https://service.geonet.org.nz
# GFZ         https://geofon.gfz.de
# ICGC        https://ws.icgc.cat
# IESDMC      http://batsws.earth.sinica.edu.tw
# IGN         http://fdsnws.sismologia.ign.es
# INGV        https://webservices.ingv.it
# IPGP        https://ws.ipgp.fr
# IRIS        https://service.earthscope.org
# IRISDMC     https://service.earthscope.org
# IRISPH5     https://service.earthscope.org
# ISC         https://www.isc.ac.uk
# KAGSR       http://sdis.emsd.ru
# KNMI        https://rdsa.knmi.nl
# KOERI       https://eida.koeri.boun.edu.tr
# LMU         https://erde.geophysik.uni-muenchen.de
# NCEDC       https://service.ncedc.org
# NIEP        https://eida-sc3.infp.ro
# NOA         https://eida.gein.noa.gr
# NRCAN       https://earthquakescanada.nrcan.gc.ca
# ODC         https://www.orfeus-eu.org
# ORFEUS      https://www.orfeus-eu.org
# RASPISHAKE  https://data.raspberryshake.org
# RESIF       https://ws.resif.fr
# SCEDC       https://service.scedc.caltech.edu
# TEXNET      http://rtserve.beg.utexas.edu
# UIB-NORSAR  https://eida.geo.uib.no
# USGS        https://earthquake.usgs.gov
# USP         https://sismo.iag.usp.br

In [4]:
def to_number(polarity):
    if polarity == 'positive':
        return 1
    elif polarity == 'negative':
        return -1
    else:
        return 0

In [5]:

# Set Client for Downloading Waveforms
client = Client("SCEDC")
event_id = 2155068
event = client.get_events(eventid=str(event_id))[0]
# big event Mw 6.7: 3144585
print(event)


Event:	1994-03-22T13:07:33.580000Z | +34.224, -118.603 | 2.3  Ml | manual

	            resource_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?eventid=2155068")
	             event_type: 'earthquake'
	          creation_info: CreationInfo(agency_id='CI', agency_uri=ResourceIdentifier(id="quakeml:doi.org/10.7909/C3WD3xH1"), creation_time=UTCDateTime(2002, 4, 11, 21, 46, 14), version='1')
	    preferred_origin_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?originid=555771")
	 preferred_magnitude_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?magnitudeid=926883")
	                   ---------
	                origins: 1 Elements
	             magnitudes: 1 Elements


In [6]:
origin = event.origins[0]
origin_time = origin.time
lat = origin.latitude
lon = origin.longitude
magnitude = event.magnitudes[0].mag

origin
# magnitude

Origin
	       resource_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?originid=555771")
	              time: UTCDateTime(1994, 3, 22, 13, 7, 33, 580000) [uncertainty=0.0]
	         longitude: -118.603 [uncertainty=0.092]
	          latitude: 34.224 [uncertainty=0.125]
	             depth: 19368.0 [uncertainty=367.0]
	        depth_type: 'from location'
	        time_fixed: False
	   epicenter_fixed: False
	         method_id: ResourceIdentifier(id="smi:ci.anss.org/origin/QED")
	       origin_type: 'hypocenter'
	   evaluation_mode: 'manual'
	 evaluation_status: 'final'
	     creation_info: CreationInfo(agency_id='CI', creation_time=UTCDateTime(2002, 4, 11, 21, 46, 14))

In [9]:
# Select Catalog around time of large event in New Zealand
# Start time is P wave arrival time according to Wilber3

starttime = UTCDateTime(1994,1,17,0,0,0)
endtime = UTCDateTime(1994,1,18,0,0,0)
catalog = client.get_events(starttime=origin_time-6000, endtime=origin_time+6000,
                            minmagnitude=magnitude-0.1, maxmagnitude=magnitude+0.1,
                            includearrivals=True)
print(catalog)
# 6.7 magnitude in the catalog
# get all stations that recorded the event
event1 = catalog[0]
print(event1)

1 Event(s) in Catalog:
1994-03-22T13:07:33.580000Z | +34.224, -118.603 | 2.3  Ml | manual
Event:	1994-03-22T13:07:33.580000Z | +34.224, -118.603 | 2.3  Ml | manual

	            resource_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?eventid=2155068")
	             event_type: 'earthquake'
	          creation_info: CreationInfo(agency_id='CI', agency_uri=ResourceIdentifier(id="quakeml:doi.org/10.7909/C3WD3xH1"), creation_time=UTCDateTime(2002, 4, 11, 21, 46, 14), version='1')
	    preferred_origin_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?originid=555771")
	 preferred_magnitude_id: ResourceIdentifier(id="quakeml:service.scedc.caltech.edu/fdsnws/event/1/query?magnitudeid=926883")
	                   ---------
	                  picks: 49 Elements
	             amplitudes: 17 Elements
	                origins: 1 Elements
	             magnitudes: 1 Elements


In [ ]:
# t1 = origin_time
# t2 = origin_time + 6000  # 10 minutes of data

# inventory = client.get_stations(
#     starttime=t1,
#     endtime=t2,   # channels must exist at event time
#     level="channel"
# )
# inventory

In [ ]:
# channels = inventory.get_contents()['channels']
# stations = inventory.get_contents()['stations']
# networks = inventory.get_contents()['networks']


In [8]:
pol_df = pd.read_csv('pol_worked.csv')
pol_df.head()

to_match = pol_df[pol_df['event_id'] == event_id]
to_match

,event_id,station,network,location,channel,p_polarity,takeoff,takeoff_uncertainty,azimuth,azimuth_uncertainty
828,2155068,IR2,CI,--,VHZ,-1.0,54.954996,0.1,50.243883,0.1
829,2155068,MWC,CI,--,BHZ,1.0,70.909062,0.1,91.759623,0.1
830,2155068,SWM,CI,--,EHZ,1.0,71.282135,0.1,3.177273,0.1
831,2155068,SIP,CI,--,VHZ,1.0,40.843851,0.1,255.803963,0.1
832,2155068,SAD,CI,--,EHZ,-1.0,45.039629,0.1,194.675821,0.1
833,2155068,PTD,CI,--,VHZ,-1.0,60.102401,0.1,214.148351,0.1
834,2155068,GRH,CI,--,ELZ,1.0,26.882078,0.1,34.131628,0.1
835,2155068,ECF,CI,--,VHZ,1.0,70.247536,0.1,299.122279,0.1
836,2155068,SLG,CI,--,EHZ,-1.0,67.662835,0.1,250.949346,0.1
837,2155068,LRR,CI,--,EHZ,-1.0,74.089596,0.1,59.309086,0.1


In [ ]:
match_net = to_match['network'].values
match_sta = to_match['station'].values
match_cha = to_match['channel'].values
match_loc = to_match['location'].values

# change all '--' to empty string
match_loc = np.where(match_loc == '--', '', match_loc)

In [ ]:
# combine net.sta.loc.cha
match_combined = []
for net, sta, loc, cha in zip(match_net, match_sta, match_loc, match_cha):
    combined = f"{net}.{sta}.{loc}.{cha}"
    match_combined.append(combined)
    
# match_combined

In [ ]:
by_channel_seedids = list(set(channels).intersection(set(match_combined)))
by_channel_seedids

In [ ]:
# split by space then first by . (stations)
by_station = [sta.split()[0].split('.')[1] for sta in stations]
# by_station

In [ ]:
matched_stations = list(set(by_station).intersection(set(match_sta)))
# matched_stations

In [ ]:
# now pick from matched_stations where to_match['station'] is in matched_stations
final_match = to_match[to_match['station'].isin(matched_stations)]
final_match

In [ ]:
by_station_seedids = []
# get network.station.''.channel from final_match
for index, row in final_match.iterrows():
    net = row['network']
    sta = row['station']
    loc = row['location'] if row['location'] != '--' else ''
    cha = row['channel']
    combined = f"{net}.{sta}.{loc}.{cha}"
    by_station_seedids.append(combined)
by_station_seedids

In [ ]:
# print lengths by station vs by channel
print("Length by station:", len(by_station_seedids))
print("Length by channel:", len(by_channel_seedids))

In [ ]:
match_sta

In [ ]:
t1

In [ ]:
# can I get waveform for matched stations? pick one and see
# start with by_channel then move to by_station

test_seedid = by_channel_seedids[1]
test_seedid
test_net =  test_seedid.split('.')[0]
test_sta =  test_seedid.split('.')[1]
test_loc =  test_seedid.split('.')[2]
test_cha =  test_seedid.split('.')[3]
# print(test_net, test_sta, test_loc, test_cha)

# starttime=UTCDateTime("2019-07-04T17:33:00"),
# endtime=UTCDateTime("2019-07-04T17:40:00")

# start=t1 and end=t2, get waveform for test_seedid
# waveform request
for seedid in by_station_seedids:
    net = seedid.split('.')[0]
    sta = seedid.split('.')[1]
    loc = seedid.split('.')[2]
    cha = seedid.split('.')[3]
# for sta in match_sta:
    # print(f"Requesting waveform for station: {sta}")
    try:
        st = client.get_waveforms(
            network=net,
            station=sta,   # example station
            location=loc,
            channel=cha,
            starttime=UTCDateTime("2019-07-04T17:33:00"),
            endtime=UTCDateTime("2019-07-04T17:40:00")
        )
        print(f"Successfully retrieved waveform for station: {sta}")
        break # testing if any station works, so break after first success
    except Exception as e:
        print(f"Failed to retrieve waveform for station: {sta}. Error: {e}")

In [ ]:
# Select Catalog around time of large event in New Zealand
# Start time is P wave arrival time according to Wilber3

starttime = UTCDateTime(1994,1,17,0,0,0)
endtime = UTCDateTime(1994,1,18,0,0,0)
catalog = client.get_events(starttime=starttime, endtime=endtime, minmagnitude=6.0,
                        maxmagnitude=7.0, includearrivals=True)
print(catalog)

In [ ]:
# 6.7 magnitude in the catalog
# get all stations that recorded the event
event0, event1 = catalog
event1

In [ ]:
[pick for pick in event1.picks if pick.phase_hint == 'P' and pick.waveform_id.station_code == 'LA02']

In [ ]:
[(pick.waveform_id.station_code, pick.waveform_id.channel_code, pick.polarity) for pick in event1.picks
  if pick.waveform_id.channel_code[0] != 'B' and pick.waveform_id.station_code == 'SMF']

In [ ]:
# plot seismogram of station

# check for what stations in example are in SCEDC
# the goal is to use overlapping stations from SKHASH

# translate data format from HASH to csv (SKHASH)

# can we use SKHASH correctly? We have to use the same stations

# find something that works from scratch...

In [ ]:
# what's the thing called... get coordinates of station

# get station coordinates
inventory = client.get_stations(network="CI", station="ABL", level="channel")
print(inventory)
# dir(inventory[0])
inventory.get_coordinates('CI.ABL..VEI')


In [ ]:
mydict = inventory.get_coordinates('CI.ABL..LHZ')
mydict
# list(mydict.values())[:2]

In [ ]:
# WR.PYR..EHZ
inventory = client.get_stations(network="WR", station="PYR", level="channel")
inventory.get_coordinates('WR.PYR..EHZ')

In [ ]:
def to_number(polarity):
    if polarity == 'positive':
        return 1
    elif polarity == 'negative':
        return -1
    else:
        return 0


# create a dictionary that stores sign for each station
data_dict = defaultdict(list)

for idx, pick in enumerate(event1.picks):
    if idx % 10 == 0:
        print(f"Processing pick {idx}/{len(event1.picks)}")
    if pick.phase_hint == 'P':
        stat = pick.waveform_id.station_code
        net = pick.waveform_id.network_code
        chan = pick.waveform_id.channel_code
        loc = pick.waveform_id.location_code
        seed_id = '.'.join([net, stat, loc, chan])
        try:
            inventory = client.get_stations(starttime=pick.time, endtime=pick.time + 1,
                                            network=net, station=stat, channel=chan, level="channel")
        except Exception as e:
            print(f"Error occurred while fetching station information for {seed_id}")
            # print error message
            print(f"Error: {e}")
            continue
        
        try:
            coordinates = inventory.get_coordinates(seed_id)
        except Exception as e:
            print(f"Error occurred while fetching coordinates for {seed_id}")
            # print error message
            print(f"Error: {e}")
            continue
        
        if len(data_dict[seed_id]) == 0:
            data_dict[seed_id].extend(list(coordinates.values())[:2] + [to_number(pick.polarity)])

# collapse to sign
for seed_id in data_dict:
    if abs(data_dict[seed_id][2]) > 1: print(f"{seed_id} has polarity {data_dict[seed_id][2]}")
    data_dict[seed_id][2] = np.sign(data_dict[seed_id][2])

In [ ]:
# NOTE: get event location
lat = event1.origins[0].latitude
lon = event1.origins[0].longitude
hdepth = event1.origins[0].depth/1000 # convert to km

col_string = 'event_id, station, network, location, channel, p_polarity, takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty'
columns = [col.strip() for col in col_string.split(',')]
df = pd.DataFrame(columns=columns)
event_id = 1
takeoff_unc = 0.1 # placeholder
az_unc = 0.1 # placeholder
epdists = []
 
velocity_model = TauPyModel(model='ak135')
# velocity_model = TauPyModel(model='iasp91')

for key, value in data_dict.items():
    net, stat, loc, chan = key.split('.')
    p_polarity = value[2]
    src2dst = [lat, lon, value[0], value[1]]
    dist, az, baz = gps2dist_azimuth(*src2dst)
    epdist = kilometers2degrees(dist / 1000) # convert to km
    
    try:
        print()
        print(f"INPUTS:\n{hdepth} km depth, {epdist:.2f} degrees distance")
        p_arrivals = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                            distance_in_degree=epdist, phase_list=['P'])
                            # phase_list=['P', 'Pn', 'Pg'])
        print(f"OUTPUTS:")
        for arrival in p_arrivals:
            print(f"Phase: {arrival.phase.name}, Time: {arrival.time:.2f} s, Takeoff angle: {arrival.takeoff_angle:.2f} degrees")
        print()
    except Exception as e:
        print(f"Error occurred while calculating travel times for station {stat}")
        # print error message
        print(f"Error: {e}")
        continue
    
    if len(p_arrivals) == 0:
        print(f"Station {stat} has no P arrival. Skipping.")
        print(f"Epidist = {epdist:.2f} degrees, depth = {hdepth} km")
        continue
    
    epdists.append(epdist)
    takeoff = p_arrivals[0].takeoff_angle
    takeoff = 180 - takeoff # convert to angle from vertical (SKHASH convention)
    
    if p_polarity != 0:
        df.loc[len(df)] = [event_id, stat, net, loc, chan, float(p_polarity),
                        takeoff, takeoff_unc, az, az_unc]
    else: print(f"Station {stat} has zero polarity. Skipping.")

In [ ]:
epdist = epdists[10]
p_arrivals = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                            distance_in_degree=epdist, phase_list=['P', 'Pn', 'Pg'])

In [ ]:
from obspy.taup import TauPyModel
import matplotlib.pyplot as plt

model = TauPyModel(model="ak135")

depth_km = 18.2
distances = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 90.0]


fig, ax = plt.subplots(
    subplot_kw=dict(polar=True),
    figsize=(8, 8)
)

for dist in distances:
    arrivals = model.get_ray_paths(
        source_depth_in_km=depth_km,
        distance_in_degree=dist,
        phase_list=["P", "Pg", "Pn"]
    )

    arrivals.plot_rays(
        fig=fig,
        ax=ax,
        show=False
    )

plt.show()

In [ ]:
hdepth

In [ ]:
model = TauPyModel(model="ak135")

fig, ax = plt.subplots(figsize=(8, 6))

max_count = 10
min_count = 0
count = 0

for epdist in epdists:
    
    arrivals = model.get_ray_paths(
        source_depth_in_km=hdepth,
        distance_in_degree=epdist,
        phase_list=["P"]
    )

    for arr in arrivals:

        path = arr.path # check it out

        x = np.degrees(path["dist"])
        y = path["depth"]

        # same but with color bar
        color = plt.cm.viridis(epdist / max(epdists))
        ax.plot(x, y, color=color)
    
    count += 1
    if count >= max_count: break
        

ax.invert_yaxis()
ax.set_xlabel("Epicentral Distance (deg)")
ax.set_ylabel("Depth (km)")
ax.legend()

plt.title("Ray Paths for P Waves")

plt.show()


In [ ]:
dir(p_arrivals[0])

In [ ]:
# NOTE: ask for a ray path diagram (TauPy function)
# NOTE: sensitivity analysis for velocity model picking

In [ ]:
# dir(velocity_model)
dir(velocity_model.model)

In [ ]:
df

In [ ]:
# save to csv
df.to_csv('event1_data.csv', index=False)